# Trabajo Final ICD 2026 - IBM HR Analytics

Objetivo: construir y comparar dos modelos supervisados para predecir si un empleado abandona la empresa (`Attrition`).

Modelos elegidos:

- Regresion Logistica
- Random Forest

## 1. Carga de librerias y dataset

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")

In [ ]:
candidate_paths = [
    Path("IBM HR Analytics Employee_TF.csv"),
    Path("../IBM HR Analytics Employee_TF.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("No se encontro el archivo IBM HR Analytics Employee_TF.csv. Subilo a Colab o dejalo junto al notebook.")

FIGURES_DIR = Path("figuras") if Path("IBM HR Analytics Employee_TF.csv").exists() else Path("../figuras")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_current_figure(filename):
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")

df = pd.read_csv(DATA_PATH, sep=";")
df.head()

## 2. Exploracion inicial

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df["Attrition"].value_counts()

In [ ]:
attrition_distribution = (
    df["Attrition"]
    .value_counts()
    .rename_axis("Attrition")
    .reset_index(name="Cantidad")
)
attrition_distribution["Porcentaje"] = (attrition_distribution["Cantidad"] / len(df) * 100).round(2)
attrition_distribution

In [ ]:
missing_values = (
    df.isna().sum()
    .reset_index(name="Faltantes")
    .rename(columns={"index": "Columna"})
)
missing_values["Porcentaje"] = (missing_values["Faltantes"] / len(df) * 100).round(2)
missing_values.query("Faltantes > 0").sort_values("Faltantes", ascending=False)

In [ ]:
constant_columns = [column for column in df.columns if df[column].nunique(dropna=False) == 1]
constant_columns

In [ ]:
df.describe().T

El dataset esta desbalanceado: hay muchos mas empleados que no abandonaron la empresa que empleados que si lo hicieron. Por eso, ademas de la exactitud (`accuracy`), se revisaran metricas como `recall` y `F1` para la clase `Yes`.

In [ ]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()
categorical_columns

## 3. Graficos exploratorios obligatorios

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Attrition", color="#4C78A8")
plt.title("Distribucion de la variable objetivo")
plt.xlabel("Rotacion laboral")
plt.ylabel("Cantidad de empleados")
save_current_figure("01_distribucion_attrition.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Attrition", y="MonthlyIncome")
plt.title("Boxplot de salario mensual segun rotacion")
plt.xlabel("Rotacion laboral")
plt.ylabel("Ingreso mensual")
save_current_figure("02_boxplot_monthly_income_attrition.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x="Attrition", y="Age")
plt.title("Violin plot de edad segun rotacion")
plt.xlabel("Rotacion laboral")
plt.ylabel("Edad")
save_current_figure("03_violin_age_attrition.png")
plt.show()

In [ ]:
def plot_attrition_rate(column, filename):
    rate = (
        df.groupby(column)["Attrition"]
        .apply(lambda values: (values == "Yes").mean() * 100)
        .sort_values(ascending=False)
        .reset_index(name="Porcentaje_Attrition_Yes")
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=rate, x=column, y="Porcentaje_Attrition_Yes", color="#4C78A8")
    plt.title(f"Porcentaje de rotacion segun {column}")
    plt.xlabel(column)
    plt.ylabel("Rotacion Yes (%)")
    plt.xticks(rotation=35, ha="right")
    save_current_figure(filename)
    plt.show()

    return rate

plot_attrition_rate("OverTime", "04_rotacion_por_overtime.png")

In [ ]:
plot_attrition_rate("JobRole", "05_rotacion_por_jobrole.png")

In [ ]:
numeric_df = df.select_dtypes(include=["int64", "float64"])

plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Mapa de calor de correlaciones numericas")
save_current_figure("06_heatmap_correlaciones.png")
plt.show()

## 4. Limpieza, preparacion y seleccion de variables

Se elimina `EmployeeNumber` porque es un identificador de empleado y no representa una caracteristica generalizable. Tambien se eliminan `EmployeeCount`, `Over18` y `StandardHours` porque tienen el mismo valor en todas las filas y no aportan informacion para clasificar.

In [ ]:
target = "Attrition"

columns_to_drop = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]

X = df.drop(columns=[target] + columns_to_drop)
y = df[target].map({"No": 0, "Yes": 1})

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features, categorical_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 5. Entrenamiento de modelos

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
    ]
)

models = {
    "Regresion Logistica": logistic_model,
    "Random Forest": random_forest_model,
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Modelo entrenado: {name}")

## 6. Evaluacion y matrices de confusion

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision_yes = precision_score(y_test, y_pred, zero_division=0)
    recall_yes = recall_score(y_test, y_pred, zero_division=0)
    f1_yes = f1_score(y_test, y_pred, zero_division=0)

    results.append(
        {
            "Modelo": name,
            "Accuracy": acc,
            "Precision_Yes": precision_yes,
            "Recall_Yes": recall_yes,
            "F1_Yes": f1_yes,
        }
    )

    print("=" * 60)
    print(name)
    print("Accuracy:", round(acc, 4))
    print(classification_report(y_test, y_pred, target_names=["No", "Yes"], zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(f"Matriz de confusion - {name}")
    plt.xlabel("Prediccion")
    plt.ylabel("Valor real")
    save_current_figure(f"07_matriz_confusion_{name.lower().replace(' ', '_')}.png")
    plt.show()

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
results_df

## 7. Comparacion final

En esta seccion se escribira la comparacion entre modelos una vez ejecutado el notebook.

In [ ]:
results_long = results_df.melt(id_vars="Modelo", var_name="Metrica", value_name="Valor")

plt.figure(figsize=(10, 5))
sns.barplot(data=results_long, x="Metrica", y="Valor", hue="Modelo")
plt.title("Comparacion de metricas entre modelos")
plt.xlabel("Metrica")
plt.ylabel("Valor")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
save_current_figure("08_comparacion_modelos.png")
plt.show()

## 8. Variables mas importantes del Random Forest

In [ ]:
feature_names = random_forest_model.named_steps["preprocessor"].get_feature_names_out()
importances = random_forest_model.named_steps["model"].feature_importances_

feature_importance = (
    pd.DataFrame({"Variable": feature_names, "Importancia": importances})
    .sort_values("Importancia", ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, y="Variable", x="Importancia", color="#4C78A8")
plt.title("Variables mas importantes segun Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Variable")
save_current_figure("09_importancia_variables_random_forest.png")
plt.show()

feature_importance